# BERT: Pre-training of Deep Bidirectional Transformers - 실습 코드 2: BERT로 텍스트 분류 파인튜닝 (SST-2 감정분석)

- Tutorial ID: `expand-bert-paper`
- Tutorial: BERT: Pre-training of Deep Bidirectional Transformers
- Section ID: `expand-bert-paper-code-2`
- Section: 실습 코드 2: BERT로 텍스트 분류 파인튜닝 (SST-2 감정분석)


In [ ]:
# ============================================================
# 코드 읽는 법 -- 실습 코드 2: BERT로 텍스트 분류 파인튜닝 (SST-2 감정분석)
#
# 이 노트북은 "실행만 하면 끝"이 아니라, BERT 논문 4장(Fine-tuning BERT)에서
# 설명한 "사전학습 + 파인튜닝" 절차가 실제 코드로는 어떻게 구현되는지
#   데이터 -> 토큰화 -> 모델 -> 학습 설정 -> 학습 -> 평가 -> 추론
# 순서를 하나씩 직접 눈으로 확인하기 위한 실습 노트입니다.
#
# 학습 목표:
#   1) 사람이 쓴 문장이 BERT가 이해할 수 있는 숫자(input_ids)로 바뀌는 과정을 직접 확인한다.
#   2) "사전학습된 BERT 본체" 위에 "새로 추가한 분류 헤드(classifier)"를 얹어
#      파인튜닝하는 구조를 이해한다.
#   3) Hugging Face의 Trainer가 학습 루프(순전파 -> loss 계산 -> 역전파 -> 파라미터 업데이트)를
#      어떻게 대신 처리해 주는지, 그리고 TrainingArguments의 각 하이퍼파라미터가
#      무엇을 조절하는지 이해한다.
#   4) 학습이 끝난 모델로 새 문장의 감정을 예측하고,
#      logits -> 확률(softmax) -> 최종 클래스(argmax)로 이어지는 해석 과정을 익힌다.
#
# 읽는 순서 (셀이 위에서부터 이 순서로 진행됩니다):
#   0단계) 실행 환경 준비 - 필요 패키지 설치, GPU 사용 가능 여부 확인
#   1단계) 데이터셋(SST-2) 구조 확인 - 어떤 입력과 정답이 주어지는가?
#   2단계) 토크나이저로 문장 1개를 직접 변환해보며 "토큰화"가 정확히 뭘 하는지 확인
#   3단계) 데이터셋 전체에 토큰화 적용 + 동적 패딩(dynamic padding) 이해
#   4단계) 분류용 BERT 모델 구조 확인 (사전학습 가중치 + 새로 추가된 분류 헤드)
#   5단계) 평가지표(accuracy) 정의
#   6단계) 학습 하이퍼파라미터(TrainingArguments) 한 줄씩 의미 확인 (+ 빠른 테스트용 서브셋 옵션)
#   7단계) Trainer 객체 생성
#   8단계) 실제 학습 실행 (Trainer.train())
#   9단계) 학습 결과(정확도) 확인
#  10단계) 학습된 모델로 새 문장 추론 + 결과 해석
#  11단계) (선택) 모델이 틀리는 예시들을 직접 살펴보는 간단한 오류 분석
#
# 주의:
#   - 이 코드는 GPU 환경(예: Colab의 T4 GPU)에서 실행하는 것을 권장합니다.
#     CPU에서는 학습이 매우 오래 걸릴 수 있습니다 (SST-2 train set은 문장 약 6만 7천 개).
#   - 처음 실행할 때는 6단계 근처의 USE_SUBSET 옵션을 True로 두고 일부 데이터로 먼저
#     "에러 없이 끝까지 도는지"를 확인한 뒤, False로 바꿔 전체 데이터로 다시 실행해 보세요.
# ============================================================


## 0단계 -- 실행 환경 준비

본격적인 코드를 보기 전에 두 가지를 먼저 준비합니다.

1. **필요한 라이브러리 설치** -- 이 노트북은 Hugging Face의 `transformers`(모델/토크나이저),
   `datasets`(데이터셋 로드), `evaluate`(평가지표 계산), `accelerate`(학습 가속 보조) 라이브러리를
   사용합니다. Colab 등 새 환경에서 처음 실행한다면 아래 셀로 한 번 설치해 주세요. 이미 설치되어
   있다면 그대로 실행해도 무방합니다(이미 있으면 빠르게 넘어갑니다).

2. **GPU 사용 가능 여부 확인** -- BERT 파인튜닝은 수많은 행렬 연산을 반복하는 작업이라
   GPU가 있으면 CPU보다 수십 배 빠릅니다. Colab을 쓴다면 메뉴에서
   `런타임 > 런타임 유형 변경 > 하드웨어 가속기`를 GPU로 설정해 주세요.


In [ ]:
# Colab이나 새로운 가상환경에서 처음 실행한다면 아래 설치 명령이 필요합니다.
# 맨 앞의 '!'는 "이 줄은 파이썬 코드가 아니라 셸(터미널) 명령어다"라는 표시입니다.
# -q 옵션은 설치 로그를 간단히(quiet) 출력하라는 의미입니다.
!pip install -q transformers datasets evaluate accelerate scikit-learn


In [ ]:
import torch

print("GPU 사용 가능 여부:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("사용 중인 GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU가 감지되지 않았습니다. CPU로도 실행은 되지만 학습이 많이 느릴 수 있습니다.")
    print("Colab이라면 [런타임] > [런타임 유형 변경]에서 GPU를 선택해 보세요.")


## 1단계 -- 데이터셋(SST-2) 살펴보기

**SST-2**(Stanford Sentiment Treebank, 2-class 버전)는 영화 리뷰에서 뽑아낸 문장에
"긍정(1)" 또는 "부정(0)" 라벨을 붙여 놓은 감정 분석용 데이터셋입니다. 여러 언어 이해
과제를 모아 둔 **GLUE 벤치마크**의 한 부분이기도 합니다.

- 입력: 영화 리뷰에서 뽑은 한 문장 (`sentence`)
- 정답: `0`(부정) 또는 `1`(긍정) (`label`)
- 데이터 분할: `train`(학습용), `validation`(검증용), `test`(테스트용)

한 가지 주의할 점이 있습니다. GLUE의 `test` 분할은 정답 라벨이 전부 `-1`로 가려져 있습니다.
GLUE 공식 리더보드에 제출해서 채점받으라는 의도이기 때문입니다. 그래서 이 실습에서는 모델
성능을 직접 확인할 수 있는 `validation` 분할을 "우리만의 시험 문제"로 사용합니다.

먼저 데이터를 불러와서 구조와 실제 예시를 눈으로 확인해 봅시다.


In [ ]:
from datasets import load_dataset

# GLUE 벤치마크 안에서 'sst2' 과제만 골라 불러옵니다.
# 처음 실행할 때는 인터넷에서 데이터를 자동으로 내려받습니다 (수십 MB 수준이라 금방 끝납니다).
dataset = load_dataset("glue", "sst2")

# -- 데이터셋 구조 확인 --
# train/validation/test로 나뉘어 있고, 각각 몇 개의 문장이 있는지 확인할 수 있습니다.
print(dataset)

# -- 실제 예시 3개 확인 --
# 'sentence': 입력 문장 / 'label': 정답(0=부정, 1=긍정) / 'idx': 문장 일련번호
print("\n예시 3개:")
for example in dataset["train"].select(range(3)):
    print(example)


## 2단계 -- 토큰화(Tokenization)란 무엇인가?

신경망은 문자가 아니라 **숫자**만 처리할 수 있습니다. 그래서 "This movie was great!" 같은
문장을 모델에 넣기 전에, 먼저 문장을 숫자로 바꿔주는 과정이 필요합니다. 이 과정을
**토큰화(tokenization)**라고 부르고, 이를 수행하는 도구가 **토크나이저(tokenizer)**입니다.

BERT는 문장을 통째로 숫자 하나로 바꾸는 것이 아니라, 다음과 같은 단계를 거칩니다.

1. 문장을 단어/부분단어(subword) 단위인 **토큰(token)**으로 쪼갭니다. (BERT는 WordPiece라는
   방식을 사용해서, 자주 쓰이는 단어는 통째로, 낯선 단어는 더 작은 조각으로 쪼갭니다.
   예: "unhappiness" -> "un", "##happiness" 처럼요.)
2. 문장 맨 앞에 **[CLS]** 토큰을, 맨 뒤에 **[SEP]** 토큰을 추가합니다.
   - `[CLS]`(classification)는 "이 문장 전체를 요약한 의미"를 담도록 학습되는 특수 토큰으로,
     뒤에서 살펴볼 분류(classification) 작업에 바로 이 토큰의 결과값을 사용합니다.
   - `[SEP]`(separator)는 문장이 끝났음을 알려주는 구분자입니다.
3. 각 토큰을 미리 정해진 단어사전(vocabulary)에 따라 고유한 정수 ID로 바꿉니다 -> 이것이 `input_ids`.
4. 여러 문장을 한 그룹으로 묶을 때 길이를 맞추기 위해 빈 자리를 `[PAD]` 토큰으로 채우는데,
   "여기는 진짜 단어다(1)" / "여기는 그냥 채운 빈칸이다(0)"를 표시하는 것이 `attention_mask`입니다.
   (지금은 문장이 하나뿐이라 패딩이 없으므로 attention_mask가 전부 1로 나옵니다.)

또한 토큰화 시 다음 옵션을 자주 사용합니다.
- `truncation=True`: 문장이 모델이 처리 가능한 최대 길이를 넘으면 잘라냅니다.
- `max_length=128`: BERT는 최대 512 토큰까지 처리할 수 있지만, SST-2는 영화 리뷰 한 문장
  정도로 짧기 때문에 128이면 충분합니다. (너무 크게 잡으면 메모리와 시간을 낭비합니다.)

말로만 설명하면 헷갈리니, 문장 하나를 직접 토큰화해서 눈으로 확인해 봅시다.


In [ ]:
from transformers import BertTokenizer

# 토크나이저를 불러옵니다.
# 'bert-base-uncased'는 사전학습된 BERT 중에서
#   - uncased: 대문자/소문자를 구분하지 않는 버전 (학습 전에 모두 소문자로 변환)
#   - base   : 작은 쪽 크기 (12개 레이어, hidden size 768) -- 더 큰 버전은 'bert-large-uncased'
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

sample_sentence = "This movie was absolutely wonderful!"
encoded = tokenizer(sample_sentence)

tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"])

print("원문 문장      :", sample_sentence)
print("토큰(token) 목록:", tokens)
print("input_ids      :", encoded["input_ids"])
print("attention_mask :", encoded["attention_mask"])

# 직접 실행해보면 다음과 같은 패턴을 확인할 수 있습니다.
#  - 토큰 목록 맨 앞/뒤에 '[CLS]'와 '[SEP]'가 자동으로 추가됨
#  - input_ids의 맨 앞 값은 101([CLS]의 ID), 맨 뒤 값은 102([SEP]의 ID) -- 이 두 숫자는 항상 고정
#  - 가운데 숫자들은 각 단어가 BERT 단어사전(vocab)에서 가지는 고유 번호
#  - attention_mask는 지금은 패딩이 없으므로 전부 1


## 3단계 -- 데이터셋 전체에 토큰화 적용 + 동적 패딩(Dynamic Padding)

먼저 짚고 갈 용어가 하나 있습니다. 모델은 보통 문장을 한 개씩 처리하지 않고 여러 개를
묶어서 한 번에 처리하는데, 이렇게 묶은 문장 그룹을 **배치(batch)**라고 부릅니다.
배치 단위로 처리하면 GPU 연산을 훨씬 효율적으로 쓸 수 있습니다.

방금은 문장 1개만 토큰화했지만, 실제로는 train/validation에 있는 수만 개 문장 전부를
토큰화해야 합니다. `dataset.map()`을 사용하면 함수 하나를 데이터셋 전체에 효율적으로
적용할 수 있습니다.

여기서 한 가지 설계 선택이 있습니다: 토큰화할 때 패딩(padding)을 **미리 하지 않습니다.**
그 이유는 다음과 같습니다.

- 만약 모든 문장을 미리 `max_length=128`로 맞춰 패딩한다면, 짧은 문장도 전부 128 길이로
  부풀어서 불필요한 연산(주로 [PAD] 토큰에 대한 연산)이 많이 발생합니다.
- 대신 **배치를 만드는 시점에**, 그 배치 안에서 "가장 긴 문장"에만 맞춰 패딩하면 훨씬
  효율적입니다. 예) 한 배치에 속한 문장 길이가 [8, 12, 5]라면, 셋 다 12로만 맞추면 충분하지
  128까지 늘릴 필요는 없습니다.

이렇게 "배치마다 그때그때" 필요한 만큼만 패딩하는 방식을 **동적 패딩(dynamic padding)**이라
하고, `DataCollatorWithPadding`이 이 역할을 자동으로 처리해 줍니다. (Trainer가 학습 중
배치를 만들 때마다 이 collator를 호출합니다.)

추가로 알아두면 좋은 점 하나: SST-2 데이터셋의 정답 컬럼 이름은 `label`(단수)인데, BERT
분류 모델이 손실(loss)을 계산할 때 내부적으로 기대하는 이름은 `labels`(복수)입니다. 이 이름
변환은 우리가 직접 코드를 작성할 필요 없이, `DataCollatorWithPadding`이 배치를 만들 때
자동으로 `label` -> `labels`로 바꿔줍니다.


In [ ]:
from transformers import DataCollatorWithPadding

def preprocess(examples):
    # map(batched=True)로 호출되므로 examples['sentence']는 문장들의 리스트입니다.
    # truncation=True, max_length=128: 128 토큰을 넘는 문장은 잘라냅니다.
    # 일부러 padding 옵션을 주지 않습니다 -- 패딩은 위에서 설명한 것처럼
    # DataCollatorWithPadding이 배치를 만들 때 처리하도록 맡깁니다.
    return tokenizer(examples["sentence"], truncation=True, max_length=128)

# .map()은 dataset의 모든 행에 preprocess 함수를 적용합니다.
# batched=True로 하면 한 번에 여러 문장을 묶어서 처리하므로 한 문장씩 처리하는 것보다 훨씬 빠릅니다.
tokenized = dataset.map(preprocess, batched=True)

# 배치를 만들 때 그때그때 패딩을 적용해주는 데이터 콜레이터를 만듭니다.
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(tokenized)
# 기존 'sentence', 'label', 'idx' 외에
# 'input_ids', 'token_type_ids', 'attention_mask' 컬럼이 새로 추가된 것을 확인할 수 있습니다.


## 4단계 -- 분류용 BERT 모델 구조 이해하기

이제 실제로 문장을 분류할 모델을 준비합니다. 핵심 아이디어는 BERT 논문의 파인튜닝 방식과
정확히 같습니다: **사전학습된 BERT 본체는 그대로 재사용하고, 그 위에 작은 분류기(classifier)
하나만 새로 얹는다.**

```
입력 문장
   ↓ (2단계에서 본 토큰화)
[CLS] this movie was great [SEP]
   ↓ (BERT 12개 레이어 통과 -- 사전학습 때 익힌 가중치를 그대로 사용)
[CLS] 토큰의 최종 hidden vector (크기 768)
   ↓ (새로 추가된 분류 헤드: 768 -> 2 크기의 작은 선형(Linear) 레이어)
[부정 점수, 긍정 점수]   ← 이 두 숫자를 "logits"라고 부릅니다
```

(여기서 hidden vector란, BERT가 각 토큰에 대해 계산해 낸 "그 토큰의 의미를 담은 숫자
벡터" 정도로 이해하면 충분합니다. [CLS] 토큰의 hidden vector는 문장 전체의 의미를
요약하도록 학습됩니다.)

`BertForSequenceClassification`을 사용하면 위 구조(BERT 본체 + 분류 헤드)가 자동으로
만들어집니다. 이때 `num_labels=2`는 "최종 출력이 2개의 클래스 점수다"라는 뜻이고, SST-2가
긍정/부정 2가지로만 나뉘는 **이진 분류(binary classification)** 문제이기 때문에 2로 둡니다.

여기서 중요한 점: BERT 본체의 가중치는 사전학습된 값을 그대로 가져오지만, 새로 추가된
분류 헤드(`classifier`)의 가중치는 아직 한 번도 학습된 적 없는 **무작위 값**으로 시작합니다.
바로 이 분류 헤드(그리고 BERT 본체 전체)를 SST-2 데이터로 학습시키는 과정이 이 노트북에서
말하는 **파인튜닝(fine-tuning)**입니다.


In [ ]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=2
)

# 모델을 불러오면 아래와 비슷한 경고(warning)가 출력될 수 있습니다 -- 정상이니 무시해도 됩니다.
#
#   Some weights of BertForSequenceClassification were not initialized from the model
#   checkpoint and are newly initialized: ['classifier.weight', 'classifier.bias']
#   You should probably TRAIN this model on a down-stream task...
#
# 뜻: "분류기(classifier) 부분은 사전학습 체크포인트에 없던 새 레이어라서 무작위 값으로
# 새로 만들었다. 그러니 이걸 학습시켜라"라는 안내입니다. 우리가 7~8단계에서 바로 그 작업을
# 할 것이므로 걱정하지 않아도 됩니다.

# 모델이 얼마나 큰지 감을 잡기 위해 전체 파라미터 개수를 세어봅니다.
num_params = sum(p.numel() for p in model.parameters())
print(f"전체 파라미터 개수: {num_params:,}")   # BERT-base 기준 약 1억 1천만 개 수준


## 5단계 -- 평가지표(accuracy) 정의하기

모델을 학습시키는 동안, "지금 모델이 얼마나 잘 맞히고 있는가"를 주기적으로 확인해야
학습이 잘 되고 있는지 판단할 수 있습니다. SST-2는 클래스가 균형 잡힌 이진 분류 문제이므로
가장 직관적인 지표인 **정확도(accuracy)** -- 전체 중 맞힌 비율 -- 를 사용합니다.

`Trainer`는 평가 시점마다 `compute_metrics` 함수를 호출해 주는데, 이 함수는

- `logits`: 모델이 두 클래스 각각에 대해 내놓은 원점수(raw score) -- shape: (샘플 수, 2)
- `labels`: 실제 정답 -- shape: (샘플 수,)

를 입력받아 "정답을 몇 % 맞혔는가"를 계산해 돌려줘야 합니다. logits 중 더 큰 값을 가지는
클래스를 모델의 예측으로 보고(`np.argmax`), 정답과 비교합니다.


In [ ]:
import numpy as np
import evaluate

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    # eval_pred는 (logits, labels) 튜플로 전달됩니다.
    logits, labels = eval_pred

    # argmax(axis=-1): 두 클래스 점수 중 더 큰 쪽의 인덱스를 예측값으로 선택합니다.
    # 예) logits = [-1.2, 2.5] -> 인덱스 1(긍정)의 점수가 더 크므로 예측은 1
    preds = np.argmax(logits, axis=-1)

    return accuracy.compute(predictions=preds, references=labels)


## 6단계 -- 학습 하이퍼파라미터(TrainingArguments) 이해하기

여기서 **하이퍼파라미터(hyperparameter)**란, 모델이 데이터를 통해 직접 학습해서 알아내는
가중치(weight)와 달리, 우리가 학습을 시작하기 전에 미리 정해서 알려주는 값들을 말합니다.
"한 번에 몇 문장씩 묶어서 학습할지", "학습을 몇 번 반복할지" 같은 것들이죠.

`TrainingArguments`는 이런 설정값들을 모아두는 객체입니다. 파인튜닝에서 자주 등장하는
하이퍼파라미터들을 하나씩 풀어서 설명합니다.

- **learning_rate (학습률)**: 한 번의 업데이트마다 파라미터를 얼마나 크게 바꿀지 정하는
  값입니다. BERT 논문은 `5e-5, 3e-5, 2e-5` 중에서 데이터셋에 맞는 값을 찾아보라고 권장합니다.
  처음부터 모델을 학습할 때보다 훨씬 작은 값을 쓰는 이유는, BERT가 사전학습 동안 이미 방대한
  언어 지식을 익혔기 때문입니다. 학습률이 너무 크면 몇 스텝만에 그 지식이 망가질 수 있습니다
  (이런 현상을 **catastrophic forgetting**이라고 부릅니다).

- **per_device_train(또는 eval)_batch_size (배치 크기)**: 한 번의 연산에 몇 개의 문장을
  묶어서(=배치로) 처리할지 정합니다. 배치가 클수록 학습이 더 안정적이고 빠르지만, GPU 메모리를
  더 많이 사용합니다. 평가(eval) 시에는 역전파(기울기 계산)가 없어 메모리 여유가 있으므로
  보통 더 크게 잡습니다.

- **num_train_epochs (에폭 수)**: 전체 학습 데이터를 몇 번 반복해서 학습할지 정합니다.
  BERT 논문은 보통 2~4 epoch이면 충분하다고 보고합니다. 너무 많이 반복하면 모델이 학습
  데이터의 사소한 패턴까지 외워버리는 **과적합(overfitting)**이 발생할 수 있습니다.

- **weight_decay (가중치 감쇠)**: 모델 파라미터(가중치)가 너무 커지지 않도록 학습 중 살짝
  억제하는 정규화 기법(L2 regularization)입니다. 과적합을 줄이는 데 도움을 줍니다.

- **warmup_ratio (워밍업 비율)**: 학습 초반 일부 구간 동안 학습률을 0에서 설정값까지 서서히
  끌어올리는 비율입니다. 0.06이면 전체 학습 스텝의 앞 6% 구간이 워밍업 구간이 됩니다.
  사전학습된 가중치에 큰 학습률을 갑자기 가하면 불안정해질 수 있어, 완만하게 시작하는 것입니다.

- **eval_strategy / save_strategy ("언제" 평가하고 저장할지)**: `"epoch"`으로 설정하면
  한 epoch이 끝날 때마다 검증 데이터로 평가하고, 그 시점의 모델을 저장합니다. (대안으로
  `"steps"`를 쓰면 일정 스텝마다 평가/저장할 수도 있습니다.) 아래에서 사용할
  `load_best_model_at_end=True` 기능이 정상 동작하려면 이 두 값이 서로 같은 주기를 가리켜야
  합니다.

- **load_best_model_at_end + metric_for_best_model**: 학습이 모두 끝난 뒤, 마지막 epoch의
  모델이 아니라 검증 성능이 "가장 좋았던" 시점의 모델을 최종적으로 사용하겠다는 설정입니다.
  "가장 좋다"의 기준은 `metric_for_best_model="accuracy"`로, 우리가 5단계에서 정의한
  정확도(accuracy)를 사용합니다. 학습 후반부에 오히려 과적합으로 성능이 떨어지는 경우를
  방지해 줍니다.


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./bert-sst2",          # 학습 중 체크포인트(모델 저장본)가 저장될 폴더

    learning_rate=2e-5,                # 파인튜닝용 학습률 (BERT 논문 권장 범위: 5e-5 ~ 2e-5)
    per_device_train_batch_size=32,    # 학습 배치 크기
    per_device_eval_batch_size=64,     # 평가 배치 크기 (역전파가 없어 더 크게 잡아도 안전)

    num_train_epochs=3,                # 전체 데이터 3회 반복 학습 (BERT 논문 권장: 2~4 epoch)
    weight_decay=0.01,                 # L2 정규화 강도

    warmup_ratio=0.06,                 # 전체 스텝의 앞 6% 구간 동안 학습률을 서서히 증가

    eval_strategy="epoch",             # 매 epoch이 끝날 때마다 validation 데이터로 평가
    save_strategy="epoch",             # 매 epoch이 끝날 때마다 체크포인트 저장
    # 참고: transformers 4.46 이전 버전에서는 evaluation_strategy 라는 이름을 썼습니다.
    #       지금 쓰고 있는 버전에서 오류가 난다면 라이브러리 버전을 확인해 보세요.

    load_best_model_at_end=True,       # 학습 종료 후, 검증 성능이 가장 좋았던 체크포인트를 사용
    metric_for_best_model="accuracy",  # "가장 좋다"의 기준 = 정확도
)


### (선택) 처음 실행한다면 -- 작은 서브셋으로 먼저 테스트해보기

SST-2의 `train` 분할은 문장이 약 6만 7천 개입니다. 전체 데이터로 3 epoch을 학습하면
GPU에서도 어느 정도 시간이 걸립니다. **코드에 오타가 있는지, 파이프라인이 끝까지 잘
도는지**부터 빠르게 확인하고 싶다면, 아래처럼 일부 데이터만 떼어서 먼저 돌려보는 것을
추천합니다. 문제없이 잘 동작하는 걸 확인한 뒤 전체 데이터로 다시 실행하면 됩니다.


In [ ]:
USE_SUBSET = True   # 처음엔 True로 빠르게 점검 -> 익숙해지면 False로 바꿔 전체 데이터로 학습

if USE_SUBSET:
    # shuffle 후 일부만 선택 (seed를 고정해 매번 같은 서브셋이 뽑히게 함)
    train_data = tokenized["train"].shuffle(seed=42).select(range(2000))
    eval_data = tokenized["validation"].shuffle(seed=42).select(range(400))
    print(f"[서브셋 모드] train {len(train_data)}개 / validation {len(eval_data)}개로 빠르게 테스트합니다.")
    print("문제없이 잘 끝나면, USE_SUBSET = False로 바꾸고 다시 실행해 전체 데이터로 학습해 보세요.")
else:
    train_data = tokenized["train"]
    eval_data = tokenized["validation"]
    print(f"[전체 데이터] train {len(train_data)}개 / validation {len(eval_data)}개로 학습합니다.")


## 7단계 -- Trainer 만들기

직접 학습 코드를 짠다면 보통 다음과 같은 반복문을 손으로 작성해야 합니다.

```
for epoch in range(num_epochs):
    for batch in train_dataloader:
        예측값 = model(batch)              # 순전파(forward)
        loss = loss_함수(예측값, 정답)        # 손실 계산
        loss.backward()                    # 역전파(backward) -- 기울기 계산
        optimizer.step()                   # 파라미터 업데이트
        optimizer.zero_grad()
    # ... 검증, 로그 기록, 체크포인트 저장 등도 직접 작성 ...
```

`Trainer`는 이 모든 과정(배치 만들기, 순전파, 손실 계산, 역전파, 옵티마이저 업데이트, 학습률
스케줄링, 주기적 평가, 체크포인트 저장 등)을 자동으로 처리해 주는 클래스입니다. 우리는
지금까지 준비한 모델/데이터/설정값/평가 함수를 `Trainer`에 전달하기만 하면 됩니다.

(참고: 분류 손실 계산에는 기본적으로 교차 엔트로피(cross-entropy) 손실 함수가, 파라미터
업데이트에는 AdamW라는 옵티마이저가 내부적으로 사용됩니다. 지금 단계에서는 "Trainer가
이런 세부 사항까지 대신 처리해 준다"는 정도만 알아도 충분합니다.)


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,          # 6-1단계에서 정한 학습 데이터 (서브셋 또는 전체)
    eval_dataset=eval_data,            # 6-1단계에서 정한 검증 데이터 (서브셋 또는 전체)
    processing_class=tokenizer,        # 토크나이저를 함께 넘기면, 모델 저장 시 같이 저장되어
                                        # 나중에 불러와 쓰기 편합니다.
                                        # (예전 버전 transformers에서는 tokenizer=tokenizer 라는
                                        #  이름의 인자를 사용했습니다 -- 현재는 processing_class 권장)
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


## 8단계 -- 학습 실행하기

이제 `trainer.train()` 한 줄이면 학습이 시작됩니다. 진행되는 동안 다음과 같은 표가 주기적으로
출력됩니다.

- `loss`: 학습 데이터에 대한 손실 값. 학습이 진행될수록 대체로 감소해야 합니다.
- `eval_loss`, `eval_accuracy`: 매 epoch이 끝날 때(`eval_strategy="epoch"`로 설정했으므로)
  validation 데이터로 평가한 손실/정확도입니다.

USE_SUBSET = True(서브셋 모드)라면 1~2분 내로, 전체 데이터로 GPU에서 학습한다면 epoch당
수 분~수십 분 정도 걸릴 수 있습니다(서브셋이 작을수록, GPU가 빠를수록 짧아집니다).


In [ ]:
trainer.train()


## 9단계 -- 최종 평가 결과 확인하기

학습이 끝나면(`load_best_model_at_end=True` 덕분에 가장 성능이 좋았던 체크포인트가 이미
모델에 로드되어 있습니다), `trainer.evaluate()`로 validation 데이터에 대한 최종 성능을
다시 한번 명시적으로 확인할 수 있습니다.

참고로 BERT 논문에서는 BERT-base 기준 SST-2에서 정확도 90% 초중반대의 성능을 보고합니다.
USE_SUBSET=True로 일부 데이터만 빠르게 돌렸다면 이보다 낮게 나올 수 있고, 전체 데이터로
제대로 학습하면 논문에 가까운 수치에 도달할 수 있습니다.


In [ ]:
eval_results = trainer.evaluate()
print(eval_results)
# 'eval_accuracy' 값이 우리가 가장 관심 있는 숫자입니다.
# (예: {'eval_loss': 0.23, 'eval_accuracy': 0.91, 'eval_runtime': ..., ...} 형태로 출력됩니다)


## 10단계 -- 학습된 모델로 새 문장 예측해보기 (추론, Inference)

이제 한 번도 보지 않은 새로운 문장을 모델에 넣어 감정을 예측해 봅니다. 이 과정에서 모델이
출력하는 값이 최종 "긍정/부정" 라벨로 바뀌는 과정을 단계별로 살펴봅니다.

1. **토큰화**: 2단계에서 했던 것과 동일하게 문장을 `input_ids`/`attention_mask`로 변환합니다.
   이번에는 `return_tensors="pt"`를 주어, 결과를 PyTorch **텐서(tensor)** 형태로 바로
   받습니다. (텐서는 PyTorch가 숫자 데이터를 다루는 기본 자료구조입니다. 지금 단계에서는
   "숫자들을 담은 배열" 정도로 이해해도 충분합니다.)
2. **순전파만 수행 (`torch.no_grad()`)**: 학습 때는 역전파를 위해 모든 연산 과정의 기울기
   (gradient)를 추적해야 하지만, 추론(예측)만 할 때는 그럴 필요가 없습니다. `torch.no_grad()`로
   감싸면 기울기 추적을 꺼서 메모리를 아끼고 속도를 높일 수 있습니다.
3. **logits -> 확률 (softmax)**: 모델이 직접 출력하는 `logits`(예: `[-1.2, 2.5]`)는 "점수"일
   뿐 확률이 아닙니다. `softmax` 함수를 적용하면 두 값의 합이 1이 되는 확률 형태
   (예: `[0.03, 0.97]`)로 바뀝니다.
4. **확률 -> 최종 클래스 (argmax)**: 두 확률 중 더 큰 쪽의 인덱스를 최종 예측 클래스로 정합니다.

추가로 한 가지 실전 팁: GPU에서 학습한 모델은 GPU 메모리에 올라가 있습니다. 토크나이저가
만든 입력은 기본적으로 CPU 텐서이므로, 모델과 입력이 서로 다른 장치(device)에 있으면 오류가
발생합니다. 그래서 입력을 모델과 같은 장치로 보내주는 `.to(model.device)`를 사용합니다.


In [ ]:
import torch

# 평가/추론 모드로 전환합니다. 학습 때만 활성화되는 dropout(일부 뉴런을 무작위로 꺼서
# 과적합을 줄이는 기법) 같은 장치들을 꺼서, 같은 입력에는 항상 같은 결과가 나오게 만듭니다.
model.eval()

label_map = {0: "Negative (부정)", 1: "Positive (긍정)"}

def predict_sentiment(text):
    # 1) 토큰화 + PyTorch 텐서로 변환 + 모델과 같은 장치(device)로 이동
    inputs = tokenizer(
        text, return_tensors="pt", truncation=True, max_length=128
    ).to(model.device)

    # 2) 추론이므로 기울기 계산을 끈 채로 순전파만 수행
    with torch.no_grad():
        logits = model(**inputs).logits   # shape: (1, 2) -> [부정 점수, 긍정 점수]

    # 3) softmax로 점수를 확률로 변환 (배치 중 첫 번째 문장이므로 [0]을 사용)
    probs = torch.softmax(logits, dim=-1)[0]

    # 4) 확률이 더 높은 클래스를 최종 예측으로 선택
    pred = torch.argmax(probs).item()
    confidence = probs[pred].item()

    return label_map[pred], confidence

# 여러 문장으로 직접 테스트해 봅니다 -- 명확한 예시와, 약간 헷갈리는 예시를 섞었습니다.
test_sentences = [
    "This movie was absolutely wonderful and inspiring!",   # 명확하게 긍정적
    "A complete waste of time, I want my money back.",      # 명확하게 부정적
    "It's not the worst film I've seen, but it's close.",   # 반어적 표현이 섞인, 헷갈리는 문장
]

for text in test_sentences:
    label, confidence = predict_sentiment(text)
    print(f"문장: {text}")
    print(f"예측: {label} (신뢰도: {confidence:.4f})\n")


## (선택) 11단계 -- 모델이 틀리는 경우 직접 들여다보기

정확도(accuracy)라는 숫자 하나만 봐서는 모델이 "어떤 유형의" 문장에서 헷갈려 하는지 알기
어렵습니다. validation 데이터 중 모델이 틀리게 예측한 문장들을 직접 모아서 살펴보면, 모델의
한계(예: 반어법, 복합적인 감정, 비유적 표현 등)를 좀 더 구체적으로 이해할 수 있습니다.

너무 오래 걸리지 않도록 validation 중 일부(200개)만 검사합니다.


In [ ]:
# 너무 오래 걸리지 않도록 validation 중 일부(200개)만 검사합니다.
sample_pool = dataset["validation"].select(range(200))

wrong_examples = []
for example in sample_pool:
    text = example["sentence"]
    true_label = example["label"]   # 0=부정, 1=긍정

    pred_label_str, confidence = predict_sentiment(text)
    pred_label = 0 if pred_label_str.startswith("Negative") else 1

    if pred_label != true_label:
        wrong_examples.append((text, label_map[true_label], pred_label_str, confidence))

print(f"검사한 {len(sample_pool)}개 문장 중 {len(wrong_examples)}개를 틀렸습니다.\n")

for text, true_label, pred_label, confidence in wrong_examples[:5]:
    print(f"문장      : {text}")
    print(f"실제 정답  : {true_label}")
    print(f"모델 예측  : {pred_label} (신뢰도: {confidence:.4f})")
    print("-" * 50)


## 정리 및 다음으로 시도해볼 것들

이번 실습에서 직접 따라가 본 전체 흐름을 다시 정리하면 다음과 같습니다.

```
원문 문장
  → 토큰화(tokenizer)                          : 단어를 숫자(input_ids)로 변환
  → 분류용 BERT(BertForSequenceClassification)  : 사전학습 BERT + 새 분류 헤드
  → Trainer + TrainingArguments                : 학습 루프와 하이퍼파라미터를 자동으로 관리
  → trainer.train()                            : 분류 헤드 + BERT 전체를 SST-2에 맞게 파인튜닝
  → 추론(logits → softmax → argmax)             : 새 문장의 감정을 예측
```

더 깊이 알아보고 싶다면 아래와 같은 실험을 직접 코드를 바꿔가며 해보는 것을 추천합니다.

- `learning_rate`를 `5e-5`, `3e-5` 등 다른 값으로 바꿔보고 성능이 어떻게 달라지는지 비교해보기
- `num_train_epochs`를 늘려보고, 어느 시점부터 validation 성능이 오히려 떨어지는지(과적합) 관찰해보기
- `bert-base-uncased` 대신 `bert-large-uncased`로 바꿔서 성능/학습 시간 차이를 비교해보기
- `trainer.state.log_history`를 이용해 학습 손실(loss) 곡선을 직접 그려보기
- 11단계에서 모은 오답 문장들의 공통점을 찾아보고, 왜 모델이 헷갈렸을지 직접 추측해보기
- 자기 자신이 쓴 문장이나 좋아하는 영화 리뷰를 `predict_sentiment()`에 넣어 테스트해보기
